# M8_8.25–M8_8.27 · Gestión de datos, estadística descriptiva y calidad

Utilizaremos los archivos compartidos del repositorio del curso:

- `data/input/pozos.csv`
- `data/input/mediciones.csv`

## Índice

### M8_8.25
- Espacio, tiempo y variables
- Unidad de observación
- Tablas, campos, registros, tipos y claves
- Datos estructurados, semiestructurados y no estructurados
- Bases relacionales y NoSQL
- Creación de tablas SQLite
- Operaciones SQL básicas
- Uniones entre tablas

### M8_8.26
- Población, muestra, variable y observación
- Recuento, mínimo, máximo y rango
- Media y mediana sin derivación de la media
- Cuantiles, IQR, varianza y desviación estándar
- Histogramas, boxplots y comparación entre grupos

### M8_8.27
- Calidad de datos
- Ausentes, duplicados, unidades, imposibles y extremos
- Datos censurados y límites de detección
- Scatter plots
- Pearson y Spearman
- Correlación y causalidad
- Dependencia espacial y temporal
- Puente a entrenamiento, validación y prueba

## Preparación: localizar el repositorio

En Google Colab, primero debe clonarse el repositorio mediante la celda facilitada en el curso.

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


print (Path.cwd())
print (Path.cwd().resolve())


In [ ]:
ROOT = Path("replace root folder path")
INPUT = ROOT / "data" / "input"
DATABASE = ROOT / "data" / "database"
OUTPUT = ROOT / "data" / "output"

DATABASE.mkdir(exist_ok=True)
(OUTPUT / "tables").mkdir(parents=True, exist_ok=True)
(OUTPUT / "figures").mkdir(parents=True, exist_ok=True)

print("Raíz del repositorio:", ROOT)

## Cargar los archivos compartidos

`pozos.csv` contiene una fila por pozo. `mediciones.csv` contiene una fila por pozo y fecha. 

In [ ]:
ruta_pozos = INPUT / "pozos.csv"
ruta_mediciones = INPUT / "mediciones.csv"

pozos = pd.read_csv(ruta_pozos)
mediciones = pd.read_csv(ruta_mediciones, parse_dates=["fecha"])

print("Dimensiones de pozos:", pozos.shape)
print("Dimensiones de mediciones:", mediciones.shape)

display(pozos.head())
display(mediciones.head())

---
# M8_8.25 · Organización y gestión de datos científicos

## 1. La naturaleza de los datos hidrogeológicos

Los datos hidrogeológicos son normalmente:

- **espaciales**, porque cada punto tiene una localización;
- **temporales**, porque muchos puntos se miden repetidamente;
- **multivariables**, porque se observan niveles, caudales, precipitación, conductividad, química y otros parámetros.

Una observación pierde parte de su significado si se separa de su localización, fecha, unidad o método de medida.

In [ ]:
# Paso 1: extensión espacial
print(pozos[["id_pozo", "x_utm", "y_utm", "acuifero"]])

# Paso 2: extensión temporal
print("Primera fecha:", mediciones["fecha"].min())
print("Última fecha:", mediciones["fecha"].max())

# Paso 3: variables observadas
columnas_identificacion = ["id_pozo", "fecha"]
variables = [c for c in mediciones.columns if c not in columnas_identificacion]
print("Variables medidas:", variables)

## 2. Unidad de observación

La **unidad de observación** es aquello que representa cada fila.

- En `pozos`, cada fila representa un pozo.
- En `mediciones`, cada fila representa una observación realizada en un pozo y una fecha.

Antes de resumir una tabla hay que preguntar: **¿qué representa exactamente una fila?** Si se mezclan unidades de observación diferentes, las estadísticas pueden dar pesos incorrectos.

In [ ]:
# Una fila por pozo
print("Número de filas en pozos:", len(pozos))
print("¿El identificador del pozo es único?", pozos["id_pozo"].is_unique)

# Varias filas por pozo
mediciones_por_pozo = mediciones.groupby("id_pozo").size()
print("Número de mediciones por pozo:")
display(mediciones_por_pozo)

## 3. Registro, campo y tipo

- **Registro:** una fila que describe una unidad de observación.
- **Campo:** una columna que representa una característica.
- **Tipo de dato:** clase de valores esperada en el campo, por ejemplo entero, decimal, texto, fecha o booleano.

El tipo tiene consecuencias. Una fecha almacenada como texto no permite trabajar correctamente con meses y años. Una columna que combina números y textos como `<0.5` necesita un tratamiento explícito.

In [ ]:
print("Tipos de la tabla pozos:")
print(pozos.dtypes)

print()
print("Tipos de la tabla mediciones:")
print(mediciones.dtypes)

## 3.1. Tipos de datos básicos en una base SQL

Al crear una tabla declaramos qué clase de contenido esperamos en cada campo. Los tipos más sencillos en SQLite son:

- **`INTEGER`**: números enteros, como un identificador numérico o el número de campaña.
- **`REAL`**: números con decimales, como una cota, concentración o profundidad.
- **`TEXT`**: texto, como `P01`, `Aluvial` o una fecha almacenada en formato ISO `AAAA-MM-DD`.
- **`NULL`**: ausencia de un valor. No es lo mismo que cero ni que una cadena vacía.

SQLite utiliza tipado flexible. Aun así, declarar tipos hace el esquema más comprensible y facilita la validación. También pueden añadirse restricciones:

- **`PRIMARY KEY`**: identifica cada registro.
- **`NOT NULL`**: obliga a proporcionar un valor.
- **`UNIQUE`**: impide repeticiones.
- **`CHECK`**: exige una condición, por ejemplo `precipitacion_mm >= 0`.
- **`FOREIGN KEY`**: conecta una tabla con otra.

In [ ]:
# Creamos una tabla mínima para observar tipos y restricciones.
conexion_tipos = sqlite3.connect(":memory:")

conexion_tipos.execute("""
CREATE TABLE muestra_tipos (
    id_muestra INTEGER PRIMARY KEY,
    codigo TEXT NOT NULL UNIQUE,
    profundidad_m REAL CHECK (profundidad_m >= 0),
    observacion TEXT
);
""")

conexion_tipos.execute("""
INSERT INTO muestra_tipos (
    id_muestra,
    codigo,
    profundidad_m,
    observacion
)
VALUES (1, 'M-001', 12.5, NULL);
""")

resultado_tipos = pd.read_sql_query(
    "SELECT * FROM muestra_tipos;",
    conexion_tipos
)

display(resultado_tipos)
conexion_tipos.close()

## 4. Claves y relaciones

- **Clave primaria:** identifica un registro de manera única.
- **Clave foránea:** contiene el valor de la clave primaria de otra tabla.
- **Clave compuesta:** utiliza dos o más campos como identificador.
- **Integridad referencial:** impide referencias a entidades que no existen.

Las relaciones más habituales son **uno a uno**, **uno a muchos** y **muchos a muchos**.

### 4.1. Relación uno a uno: `1 : 1`

Cada registro de la primera tabla se relaciona, como máximo, con un registro de la segunda, y viceversa.

```text
pozos          construccion_pozo
  P01     1  ───────────  1     P01
```

Ejemplo: cada pozo tiene una única ficha constructiva vigente. En la segunda tabla, `id_pozo` es simultáneamente clave primaria y clave foránea.

In [ ]:
conexion_relaciones = sqlite3.connect(":memory:")
conexion_relaciones.execute("PRAGMA foreign_keys = ON;")

conexion_relaciones.execute("""
CREATE TABLE pozos_rel (
    id_pozo TEXT PRIMARY KEY,
    nombre TEXT NOT NULL
);
""")

conexion_relaciones.execute("""
CREATE TABLE construccion_pozo (
    id_pozo TEXT PRIMARY KEY,
    profundidad_total_m REAL,
    diametro_mm REAL,
    FOREIGN KEY (id_pozo) REFERENCES pozos_rel(id_pozo)
);
""")

conexion_relaciones.execute(
    "INSERT INTO pozos_rel VALUES ('P01', 'Pozo norte');"
)
conexion_relaciones.execute(
    "INSERT INTO construccion_pozo VALUES ('P01', 45.0, 200.0);"
)

consulta_uno_a_uno = """
SELECT
    p.id_pozo,
    p.nombre,
    c.profundidad_total_m,
    c.diametro_mm
FROM pozos_rel AS p
INNER JOIN construccion_pozo AS c
    ON p.id_pozo = c.id_pozo;
"""

display(pd.read_sql_query(consulta_uno_a_uno, conexion_relaciones))

### 4.2. Relación uno a muchos: `1 : N`

Un registro de la primera tabla puede relacionarse con varios registros de la segunda. Cada registro de la segunda pertenece a un único registro de la primera.

```text
pozos                 mediciones_rel
  P01      1  ───────────  N    varias fechas
```

Ejemplo: un pozo puede tener muchas mediciones. La clave foránea se sitúa en el lado “muchos”, es decir, en `mediciones_rel`.

In [ ]:
conexion_relaciones.execute("""
CREATE TABLE mediciones_rel (
    id_medicion INTEGER PRIMARY KEY,
    id_pozo TEXT NOT NULL,
    fecha TEXT NOT NULL,
    nivel_m REAL,
    FOREIGN KEY (id_pozo) REFERENCES pozos_rel(id_pozo)
);
""")

conexion_relaciones.executemany(
    """
    INSERT INTO mediciones_rel (
        id_medicion,
        id_pozo,
        fecha,
        nivel_m
    )
    VALUES (?, ?, ?, ?);
    """,
    [
        (1, "P01", "2025-01-01", 8.7),
        (2, "P01", "2025-02-01", 8.4),
        (3, "P01", "2025-03-01", 8.1)
    ]
)

consulta_uno_a_muchos = """
SELECT
    p.id_pozo,
    p.nombre,
    m.fecha,
    m.nivel_m
FROM pozos_rel AS p
INNER JOIN mediciones_rel AS m
    ON p.id_pozo = m.id_pozo
ORDER BY m.fecha;
"""

display(pd.read_sql_query(consulta_uno_a_muchos, conexion_relaciones))

### 4.3. Relación muchos a muchos: `N : M`

Varios registros de la primera tabla pueden relacionarse con varios registros de la segunda.

```text
pozos         pozo_programa         programas
  N       ─── tabla intermedia ───      M
```

Ejemplo: un pozo puede participar en varios programas de seguimiento y cada programa puede incluir varios pozos.

Una relación muchos a muchos se representa mediante una **tabla intermedia**. La clave primaria de la tabla intermedia suele ser compuesta: `(id_pozo, id_programa)`.

In [ ]:
conexion_relaciones.execute("""
CREATE TABLE programas (
    id_programa INTEGER PRIMARY KEY,
    nombre_programa TEXT NOT NULL
);
""")

conexion_relaciones.execute("""
CREATE TABLE pozo_programa (
    id_pozo TEXT NOT NULL,
    id_programa INTEGER NOT NULL,
    fecha_alta TEXT,
    PRIMARY KEY (id_pozo, id_programa),
    FOREIGN KEY (id_pozo) REFERENCES pozos_rel(id_pozo),
    FOREIGN KEY (id_programa) REFERENCES programas(id_programa)
);
""")

# Añadimos un segundo pozo y dos programas.
conexion_relaciones.execute(
    "INSERT INTO pozos_rel VALUES ('P02', 'Pozo sur');"
)
conexion_relaciones.executemany(
    "INSERT INTO programas VALUES (?, ?);",
    [
        (1, "Control de niveles"),
        (2, "Control de calidad")
    ]
)

# P01 participa en dos programas; P02 participa en uno.
conexion_relaciones.executemany(
    "INSERT INTO pozo_programa VALUES (?, ?, ?);",
    [
        ("P01", 1, "2025-01-01"),
        ("P01", 2, "2025-01-01"),
        ("P02", 1, "2025-02-01")
    ]
)

consulta_muchos_a_muchos = """
SELECT
    p.id_pozo,
    p.nombre,
    g.nombre_programa,
    pp.fecha_alta
FROM pozo_programa AS pp
INNER JOIN pozos_rel AS p
    ON pp.id_pozo = p.id_pozo
INNER JOIN programas AS g
    ON pp.id_programa = g.id_programa
ORDER BY p.id_pozo, g.id_programa;
"""

display(pd.read_sql_query(consulta_muchos_a_muchos, conexion_relaciones))
conexion_relaciones.close()

### 4.4. Resumen de relaciones

- **Uno a uno:** la segunda tabla amplía una entidad sin repetirla.
- **Uno a muchos:** la clave foránea se coloca en la tabla del lado “muchos”.
- **Muchos a muchos:** se crea una tabla intermedia que contiene las dos claves foráneas.

La cardinalidad se decide a partir del significado de los datos, no de la apariencia de los archivos.

In [ ]:
# Comprobar la clave primaria propuesta
print("¿id_pozo es único en pozos?", pozos["id_pozo"].is_unique)

# Comprobar integridad referencial
ids_en_mediciones = set(mediciones["id_pozo"])
ids_en_pozos = set(pozos["id_pozo"])
ids_sin_pozo = ids_en_mediciones - ids_en_pozos
print("Identificadores de mediciones sin pozo asociado:", ids_sin_pozo)

# Comprobar la clave lógica pozo-fecha
repetidas = mediciones.duplicated(["id_pozo", "fecha"], keep=False)
print("Filas con pozo-fecha repetido:", repetidas.sum())
display(mediciones.loc[repetidas].sort_values(["id_pozo", "fecha"]))

## 5. Datos estructurados, semiestructurados y no estructurados

- **Estructurados:** tienen columnas y tipos definidos, como CSV y tablas SQL.
- **Semiestructurados:** tienen etiquetas o jerarquías flexibles, como JSON y XML.
- **No estructurados:** no tienen inicialmente una estructura tabular, como informes, fotografías o imágenes de testigos.

“No estructurado” no significa “sin información”. Significa que la estructura debe extraerse antes del análisis.

## 6. Bases relacionales y NoSQL

Una **base relacional** organiza datos en tablas conectadas y utiliza SQL. Conviene cuando importan las relaciones, las restricciones y las consultas reproducibles.

**NoSQL** incluye familias diferentes: documentos, clave-valor, grafos y columnas anchas. No es necesariamente mejor ni peor. Responde a otras estructuras y escalas.

- **PostgreSQL:** gestor relacional robusto y multiusuario.
- **PostGIS:** extensión espacial de PostgreSQL.
- **SQLite:** base relacional contenida en un solo archivo.
- **GeoPackage:** formato geoespacial basado en SQLite.

Utilizaremos SQLite porque permite aprender tablas, tipos, claves y SQL sin administrar un servidor.

## 7. Crear una base y tablas sencillas

Primero veremos una base mínima. `CREATE TABLE` define la estructura antes de insertar datos. Esto permite declarar tipos, una clave primaria y una clave foránea.

In [ ]:
ruta_ejemplo = DATABASE / "ejemplo_sql.sqlite"

# Abrimos una conexión. Si el archivo no existe, SQLite lo crea.
conexion = sqlite3.connect(ruta_ejemplo)

# Activamos la comprobación de claves foráneas en SQLite.
conexion.execute("PRAGMA foreign_keys = ON;")

# Borramos las tablas del ejemplo si ya existen.
conexion.execute("DROP TABLE IF EXISTS mediciones_ejemplo;")
conexion.execute("DROP TABLE IF EXISTS pozos_ejemplo;")

# Creamos una tabla sencilla de pozos.
conexion.execute("""
CREATE TABLE pozos_ejemplo (
    id_pozo TEXT PRIMARY KEY,
    acuifero TEXT NOT NULL,
    cota_terreno_m REAL
);
""")

# Creamos una tabla de mediciones relacionada con la anterior.
conexion.execute("""
CREATE TABLE mediciones_ejemplo (
    id_medicion INTEGER PRIMARY KEY,
    id_pozo TEXT NOT NULL,
    fecha TEXT NOT NULL,
    nivel_m REAL,
    FOREIGN KEY (id_pozo) REFERENCES pozos_ejemplo(id_pozo)
);
""")

conexion.commit()
print("Tablas creadas.")

## 8. Insertar filas

`INSERT INTO` añade registros. Los nombres de las columnas se escriben explícitamente para que la operación sea legible y menos dependiente del orden físico de la tabla.

In [ ]:
# Insertamos dos pozos.
conexion.execute("""
INSERT INTO pozos_ejemplo (id_pozo, acuifero, cota_terreno_m)
VALUES ('P01', 'Aluvial', 112.4);
""")

conexion.execute("""
INSERT INTO pozos_ejemplo (id_pozo, acuifero, cota_terreno_m)
VALUES ('P02', 'Carbonatado', 176.2);
""")

# Insertamos tres mediciones.
conexion.execute("""
INSERT INTO mediciones_ejemplo (id_pozo, fecha, nivel_m)
VALUES ('P01', '2025-01-01', 8.7);
""")
conexion.execute("""
INSERT INTO mediciones_ejemplo (id_pozo, fecha, nivel_m)
VALUES ('P01', '2025-02-01', 8.4);
""")
conexion.execute("""
INSERT INTO mediciones_ejemplo (id_pozo, fecha, nivel_m)
VALUES ('P02', '2025-01-01', 21.5);
""")

conexion.commit()
print("Filas insertadas.")

## 9. SELECT: consultar columnas

`SELECT` elige columnas y `FROM` indica la tabla. Se recomienda evitar `SELECT *` en consultas definitivas cuando solo se requieren algunos campos.

In [ ]:
consulta = """
SELECT id_pozo, acuifero
FROM pozos_ejemplo;
"""

resultado = pd.read_sql_query(consulta, conexion)
display(resultado)

## 10. WHERE: filtrar filas

`WHERE` conserva únicamente las filas que cumplen una condición.

In [ ]:
consulta = """
SELECT id_pozo, fecha, nivel_m
FROM mediciones_ejemplo
WHERE nivel_m > 10;
"""

resultado = pd.read_sql_query(consulta, conexion)
display(resultado)

## 11. ORDER BY y operaciones calculadas

`ORDER BY` ordena el resultado. SQL también puede calcular expresiones sencillas. La cota piezométrica se obtiene restando la profundidad del nivel a la cota del terreno, pero para ello necesitamos unir las dos tablas.

## 12. Qué hace un JOIN

Un `JOIN` combina filas relacionadas.

### Tabla de pozos

| id_pozo | acuifero | cota_terreno_m |
|---|---|---:|
| P01 | Aluvial | 112.4 |
| P02 | Carbonatado | 176.2 |

### Tabla de mediciones

| id_pozo | fecha | nivel_m |
|---|---|---:|
| P01 | 2025-01-01 | 8.7 |
| P01 | 2025-02-01 | 8.4 |
| P02 | 2025-01-01 | 21.5 |

El campo común es `id_pozo`. El resultado repite la información del pozo para cada medición relacionada.

In [ ]:
consulta = """
SELECT
    m.id_pozo,
    m.fecha,
    p.acuifero,
    p.cota_terreno_m,
    m.nivel_m,
    p.cota_terreno_m - m.nivel_m AS cota_piezometrica_m
FROM mediciones_ejemplo AS m
INNER JOIN pozos_ejemplo AS p
    ON m.id_pozo = p.id_pozo
ORDER BY m.id_pozo, m.fecha;
"""

resultado_join = pd.read_sql_query(consulta, conexion)
display(resultado_join)

## 13. INNER JOIN y LEFT JOIN

- **INNER JOIN:** devuelve solo filas con correspondencia en ambas tablas.
- **LEFT JOIN:** devuelve todas las filas de la tabla izquierda, aunque no exista correspondencia a la derecha.

`LEFT JOIN` es útil para detectar mediciones sin metadatos o pozos sin una clasificación completa.

In [ ]:
# Añadimos una tabla auxiliar de responsables con información incompleta.
conexion.execute("DROP TABLE IF EXISTS responsables_ejemplo;")
conexion.execute("""
CREATE TABLE responsables_ejemplo (
    id_pozo TEXT PRIMARY KEY,
    responsable TEXT
);
""")
conexion.execute("""
INSERT INTO responsables_ejemplo (id_pozo, responsable)
VALUES ('P01', 'Equipo A');
""")
conexion.commit()

consulta_inner = """
SELECT p.id_pozo, p.acuifero, r.responsable
FROM pozos_ejemplo AS p
INNER JOIN responsables_ejemplo AS r
    ON p.id_pozo = r.id_pozo;
"""

consulta_left = """
SELECT p.id_pozo, p.acuifero, r.responsable
FROM pozos_ejemplo AS p
LEFT JOIN responsables_ejemplo AS r
    ON p.id_pozo = r.id_pozo;
"""

print("INNER JOIN")
display(pd.read_sql_query(consulta_inner, conexion))

print("LEFT JOIN")
display(pd.read_sql_query(consulta_left, conexion))

## 14. COUNT, AVG y GROUP BY

Las funciones agregadas resumen varias filas:

- `COUNT`: cuenta valores no nulos;
- `AVG`: calcula la media;
- `MIN` y `MAX`: extremos;
- `SUM`: suma.

`GROUP BY` define los grupos. Debemos saber qué representa cada grupo y si las observaciones son comparables.

In [ ]:
consulta = """
SELECT
    id_pozo,
    COUNT(nivel_m) AS numero_mediciones,
    AVG(nivel_m) AS nivel_medio,
    MIN(nivel_m) AS nivel_minimo,
    MAX(nivel_m) AS nivel_maximo
FROM mediciones_ejemplo
GROUP BY id_pozo
ORDER BY id_pozo;
"""

resumen_sql = pd.read_sql_query(consulta, conexion)
display(resumen_sql)
conexion.close()

## 15. Crear la base del curso desde los CSV

Ahora aplicamos la misma lógica a los archivos compartidos. En lugar de `to_sql`, creamos las tablas explícitamente y después insertamos las filas. Esto hace visibles el esquema, las claves y las restricciones.

In [ ]:
ruta_db_curso = DATABASE / "hidrogeologia.sqlite"
conexion = sqlite3.connect(ruta_db_curso)
conexion.execute("PRAGMA foreign_keys = ON;")

conexion.execute("DROP TABLE IF EXISTS mediciones;")
conexion.execute("DROP TABLE IF EXISTS pozos;")

conexion.execute("""
CREATE TABLE pozos (
    id_pozo TEXT PRIMARY KEY,
    acuifero TEXT NOT NULL,
    x_utm REAL NOT NULL,
    y_utm REAL NOT NULL,
    cota_terreno_m REAL
);
""")

conexion.execute("""
CREATE TABLE mediciones (
    id_medicion INTEGER PRIMARY KEY AUTOINCREMENT,
    id_pozo TEXT NOT NULL,
    fecha TEXT NOT NULL,
    profundidad_nivel_m REAL,
    precipitacion_mm REAL,
    conductividad_uScm REAL,
    FOREIGN KEY (id_pozo) REFERENCES pozos(id_pozo)
);
""")

conexion.commit()
print("Esquema creado.")

In [ ]:
# Preparamos las filas como listas de tuplas.
filas_pozos = list(
    pozos[["id_pozo", "acuifero", "x_utm", "y_utm", "cota_terreno_m"]]
    .itertuples(index=False, name=None)
)

filas_mediciones = list(
    mediciones.assign(fecha=mediciones["fecha"].dt.strftime("%Y-%m-%d"))[
        ["id_pozo", "fecha", "profundidad_nivel_m", "precipitacion_mm", "conductividad_uScm"]
    ].itertuples(index=False, name=None)
)

# executemany ejecuta la misma sentencia para muchas filas.
conexion.executemany("""
INSERT INTO pozos (id_pozo, acuifero, x_utm, y_utm, cota_terreno_m)
VALUES (?, ?, ?, ?, ?);
""", filas_pozos)

conexion.executemany("""
INSERT INTO mediciones (
    id_pozo, fecha, profundidad_nivel_m, precipitacion_mm, conductividad_uScm
)
VALUES (?, ?, ?, ?, ?);
""", filas_mediciones)

conexion.commit()
print("Pozos insertados:", len(filas_pozos))
print("Mediciones insertadas:", len(filas_mediciones))

## 16. Consulta hidrogeológica completa

La consulta une mediciones y pozos, filtra un acuífero y calcula la cota piezométrica. Cada línea cumple una función visible.

In [ ]:
consulta = """
SELECT
    m.id_pozo,
    m.fecha,
    p.acuifero,
    m.profundidad_nivel_m,
    p.cota_terreno_m,
    p.cota_terreno_m - m.profundidad_nivel_m AS cota_piezometrica_m
FROM mediciones AS m
INNER JOIN pozos AS p
    ON m.id_pozo = p.id_pozo
WHERE p.acuifero = 'Aluvial'
ORDER BY m.id_pozo, m.fecha;
"""

niveles_aluvial = pd.read_sql_query(
    consulta,
    conexion,
    parse_dates=["fecha"]
)

display(niveles_aluvial.head(10))
conexion.close()

### Actividad M8_8.25

1. Crea una consulta que devuelva únicamente `id_pozo`, `fecha` y `conductividad_uScm`.
2. Añade un filtro para conductividad superior a 1.000 µS/cm.
3. Ordena de mayor a menor.
4. Une `mediciones` con `pozos` para añadir el acuífero.
5. Explica qué filas desaparecerían con `INNER JOIN` y cuáles se conservarían con `LEFT JOIN`.

---
# M8_8.26 · Estadística descriptiva

## 17. Población, muestra, observación y variable

- **Población:** conjunto completo sobre el que se desea conocer algo.
- **Muestra:** observaciones disponibles.
- **Observación:** valor registrado para una unidad.
- **Variable:** característica medida.
- **Parámetro:** propiedad numérica de la población.

## 18. Recuento, mínimo, máximo y rango

El rango es:

$$
R=x_{max}-x_{min}
$$

Es fácil de interpretar, pero depende solo de dos observaciones y es sensible a errores y extremos.

In [ ]:
serie = mediciones["conductividad_uScm"]

numero_filas = len(serie)
numero_validos = serie.count()
minimo = serie.min()
maximo = serie.max()
rango = maximo - minimo

print("Filas:", numero_filas)
print("Valores válidos:", numero_validos)
print("Mínimo:", minimo)
print("Máximo:", maximo)
print("Rango:", rango)

## 19. Media y mediana

La **media** utiliza todos los valores y representa un punto de equilibrio numérico. Es sensible a valores extremos.

La **mediana** es el valor central de los datos ordenados. Es más resistente a extremos.

Para `[8, 8, 9, 9, 41]`, la media es 15 y la mediana es 9. No existe una medida universalmente mejor: la elección depende de la distribución y la pregunta.

In [ ]:
ejemplo = pd.Series([8, 8, 9, 9, 41])

print("Datos:", ejemplo.to_list())
print("Media:", ejemplo.mean())
print("Mediana:", ejemplo.median())

## 20. Cuantiles e intervalo intercuartílico

Un cuantil $Q_p$ deja aproximadamente una proporción $p$ de los datos por debajo.

- $Q_1$: 25 %;
- $Q_2$: mediana;
- $Q_3$: 75 %.

$$
IQR=Q_3-Q_1
$$

El IQR describe la mitad central y es más resistente a extremos que el rango. La regla de 1,5 IQR en un boxplot identifica puntos alejados, no errores confirmados.

In [ ]:
q1 = serie.quantile(0.25)
mediana = serie.quantile(0.50)
q3 = serie.quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print("Q1:", q1)
print("Mediana:", mediana)
print("Q3:", q3)
print("IQR:", iqr)
print("Límites gráficos:", limite_inferior, limite_superior)

## 21. Varianza y desviación estándar

La **varianza** y la **desviación estándar** describen la dispersión de los valores alrededor de la media. Ambas utilizan las diferencias entre cada observación y la media, pero se expresan de manera distinta.

### Varianza muestral

La varianza muestral se calcula como:

$$
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2
$$

donde:

- $x_i$ es cada observación;
- $\bar{x}$ es la media de las observaciones;
- $n$ es el número de observaciones válidas;
- $x_i-\bar{x}$ es la desviación de cada observación respecto a la media;
- $n-1$ corresponde a los grados de libertad utilizados para estimar la varianza a partir de una muestra.

Las desviaciones se elevan al cuadrado para evitar que las desviaciones positivas y negativas se cancelen.

La varianza se expresa en las unidades de la variable elevadas al cuadrado. Si la profundidad se expresa en metros, la varianza se expresa en metros cuadrados, $m^2$.

### Desviación estándar

La desviación estándar es la raíz cuadrada de la varianza:

$$
s = \sqrt{s^2}
$$

La desviación estándar se expresa en las mismas unidades que la variable original. Por ejemplo, si la profundidad se mide en metros, la desviación estándar también se expresa en metros.

### Diferencia conceptual

- La **varianza** representa la dispersión utilizando desviaciones al cuadrado.
- La **desviación estándar** expresa esa dispersión en las unidades originales de la variable.
- La varianza resulta especialmente útil en cálculos estadísticos y modelos matemáticos.
- La desviación estándar suele ser más fácil de interpretar y comunicar.

Por ejemplo, una varianza de $36\ m^2$ corresponde a una desviación estándar de:

$$
s = \sqrt{36\ m^2} = 6\ m
$$

La desviación estándar puede interpretarse como una distancia característica de los valores respecto a la media, aunque esta interpretación debe hacerse con prudencia cuando la distribución es asimétrica, contiene valores extremos o combina grupos diferentes.

In [ ]:
# Seleccionamos la variable y eliminamos únicamente los valores ausentes.
profundidad = mediciones["profundidad_nivel_m"].dropna()

# Calculamos los estadísticos descriptivos.
media = profundidad.mean()
varianza = profundidad.var()
desviacion_estandar = profundidad.std()

# Calculamos los límites situados a una desviación estándar de la media.
limite_inferior = media - desviacion_estandar
limite_superior = media + desviacion_estandar

# Mostramos los resultados numéricos.
print(f"Número de observaciones válidas: {profundidad.count()}")
print(f"Media: {media:.2f} m")
print(f"Varianza muestral: {varianza:.2f} m²")
print(f"Desviación estándar muestral: {desviacion_estandar:.2f} m")
print(
    "Intervalo media ± una desviación estándar: "
    f"{limite_inferior:.2f} a {limite_superior:.2f} m"
)

# Creamos el histograma.
figura, eje = plt.subplots(figsize=(10, 5))

eje.hist(
    profundidad,
    bins=12,
    edgecolor="black",
    alpha=0.7
)

# Marcamos la media.
eje.axvline(
    media,
    color="black",
    linewidth=2.5,
    label=f"Media = {media:.2f} m"
)

# Marcamos media - una desviación estándar.
eje.axvline(
    limite_inferior,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label=f"Media − 1 s = {limite_inferior:.2f} m"
)

# Marcamos media + una desviación estándar.
eje.axvline(
    limite_superior,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label=f"Media + 1 s = {limite_superior:.2f} m"
)

# Sombreamos el intervalo media ± una desviación estándar.
eje.axvspan(
    limite_inferior,
    limite_superior,
    color="tab:orange",
    alpha=0.15,
    label="Intervalo media ± 1 s"
)

# Añadimos la varianza como anotación.
# La varianza no se representa como una línea porque se expresa en m²,
# mientras que el eje horizontal se expresa en metros.
texto_varianza = f"Varianza = {varianza:.2f} m²"

eje.text(
    0.98,
    0.95,
    texto_varianza,
    transform=eje.transAxes,
    horizontalalignment="right",
    verticalalignment="top",
    bbox={
        "boxstyle": "round",
        "facecolor": "white",
        "edgecolor": "gray",
        "alpha": 0.9
    }
)

# Completamos la figura.
eje.set_title(
    "Distribución de la profundidad del nivel y medidas de dispersión"
)
eje.set_xlabel("Profundidad del nivel (m)")
eje.set_ylabel("Número de observaciones")
eje.legend()

plt.tight_layout()
plt.show()

## 22. Histogramas y boxplots

El histograma muestra cómo se distribuyen las observaciones en intervalos. Su apariencia depende del número de intervalos.

El boxplot representa mediana, cuartiles, IQR, bigotes y puntos alejados. Facilita comparaciones, pero oculta parte de la forma y del tamaño muestral.

In [ ]:
figura, ejes = plt.subplots(1, 2, figsize=(11, 4))

ejes[0].hist(serie.dropna(), bins=15, edgecolor="black")
ejes[0].set_title("Distribución de la conductividad")
ejes[0].set_xlabel("Conductividad (µS/cm)")
ejes[0].set_ylabel("Número de observaciones")

ejes[1].boxplot(serie.dropna())
ejes[1].set_title("Boxplot de la conductividad")
ejes[1].set_ylabel("Conductividad (µS/cm)")

plt.tight_layout()
plt.show()

## 23. Comparación entre grupos

Un resumen global puede mezclar sistemas distintos. Se comparan grupos con significado hidrogeológico, como acuíferos, pozos o campañas.

Antes de interpretar diferencias:

- comprobar unidades y métodos;
- comprobar periodos comparables;
- revisar tamaños de grupo;
- reconocer observaciones repetidas;
- distinguir descripción de inferencia.

In [ ]:
datos = mediciones.merge(
    pozos,
    on="id_pozo",
    how="left",
    validate="many_to_one"
)

resumen_por_acuifero = datos.groupby("acuifero")["conductividad_uScm"].agg(
    numero_validos="count",
    media="mean",
    mediana="median",
    minimo="min",
    maximo="max",
    desviacion_estandar="std"
)

display(resumen_por_acuifero.round(2))

### Actividad M8_8.26

Calcula y representa los principales resúmenes de `profundidad_nivel_m`. Después, compara los acuíferos y explica por qué el resumen global puede resultar engañoso.

---
# M8_8.27 · Calidad y relaciones entre variables

## 24. Dimensiones de calidad

La calidad siempre se evalúa respecto a un uso. Un dato puede ser válido para un inventario y no ser adecuado para comparar campañas.

## 25. Ausentes, duplicados, imposibles y valores que requieren revisión

- Ausente no es lo mismo que cero.
- Un duplicado exacto repite todos los campos.
- Un duplicado lógico repite la clave esperada.
- Un imposible viola una restricción física o definicional.
- Un valor extremo puede ser error o evento real.

La estadística señala casos; la decisión requiere metadatos y conocimiento hidrogeológico.

In [ ]:
numero_ausentes = mediciones["profundidad_nivel_m"].isna().sum()
numero_duplicados = mediciones.duplicated().sum()
numero_precipitaciones_negativas = (mediciones["precipitacion_mm"] < 0).sum()
numero_conductividades_altas = (mediciones["conductividad_uScm"] > 3000).sum()

print("Niveles ausentes:", numero_ausentes)
print("Filas duplicadas:", numero_duplicados)
print("Precipitaciones negativas:", numero_precipitaciones_negativas)
print("Conductividades > 3000 µS/cm:", numero_conductividades_altas)

## 26. Tratamiento no destructivo

Los originales no se sobrescriben. Se crea una copia de análisis y se registra cada decisión. Las banderas permiten conservar valores pendientes de revisión.

In [ ]:
datos_originales = mediciones.copy()
datos_analisis = datos_originales.drop_duplicates().copy()

# Una precipitación mensual negativa es físicamente imposible.
datos_analisis.loc[
    datos_analisis["precipitacion_mm"] < 0,
    "precipitacion_mm"
] = np.nan

# La conductividad alta se conserva, pero se marca para revisión.
datos_analisis["revisar_conductividad"] = (
    datos_analisis["conductividad_uScm"] > 3000
)

print("Forma original:", datos_originales.shape)
print("Forma de análisis:", datos_analisis.shape)
display(datos_analisis.loc[datos_analisis["revisar_conductividad"]])

## 28. Diagramas de dispersión

Un scatter plot muestra pares $(x_i,y_i)$. Debemos observar dirección, forma, dispersión, grupos y valores influyentes antes de calcular un coeficiente.

In [ ]:
analisis = datos_analisis.merge(
    pozos,
    on="id_pozo",
    how="left",
    validate="many_to_one"
)

figura, eje = plt.subplots(figsize=(7, 5))

for acuifero, grupo in analisis.groupby("acuifero"):
    eje.scatter(
        grupo["profundidad_nivel_m"],
        grupo["conductividad_uScm"],
        label=acuifero,
        alpha=0.7
    )

eje.set_xlabel("Profundidad del nivel (m)")
eje.set_ylabel("Conductividad (µS/cm)")
eje.set_title("Profundidad y conductividad")
eje.legend(title="Acuífero")
plt.show()

## 29. Pearson y Spearman

### Pearson

Mide asociación lineal:

$$
r=\frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum(x_i-\bar{x})^2}\sqrt{\sum(y_i-\bar{y})^2}}
$$

### Spearman

Calcula la correlación sobre rangos y mide asociación monotónica. Sin empates:

$$
ho=1-\frac{6\sum d_i^2}{n(n^2-1)}
$$

Un valor próximo a cero no demuestra independencia. Pearson es especialmente sensible a puntos influyentes. Ninguno corrige mezcla de grupos o dependencia.

In [ ]:
variables_relacion = analisis[[
    "profundidad_nivel_m",
    "conductividad_uScm"
]]

pearson = variables_relacion.corr(method="pearson")
spearman = variables_relacion.corr(method="spearman")

print("Pearson")
display(pearson)

print("Spearman")
display(spearman)

## 30. Correlación no implica causalidad

Una asociación entre profundidad y conductividad puede reflejar acuífero, litología, tiempo de residencia, profundidad constructiva, periodo de muestreo, bombeo o una observación influyente.

La causalidad requiere un mecanismo físico defendible, secuencia temporal y evidencia adicional.

## 33. Entrenamiento, validación y prueba

- **Entrenamiento:** ajusta el modelo.
- **Validación:** ayuda a elegir métodos y configuraciones.
- **Prueba:** evalúa el resultado final.

Si meses del mismo pozo aparecen en entrenamiento y prueba, la evaluación puede parecer mejor de lo que sería en un pozo nuevo. Según la pregunta, conviene separar por pozo, zona, campaña o periodo.
(Overfitting & Underfitting) 
### Underfitting, ajuste adecuado y overfitting

- **Underfitting o subajuste:** el modelo es demasiado simple para representar el patrón principal de los datos. El error suele ser alto tanto en entrenamiento como en validación.
- **Ajuste adecuado:** el modelo representa el patrón general sin intentar reproducir cada pequeña variación de los datos de entrenamiento.
- **Overfitting o sobreajuste:** el modelo sigue demasiado de cerca los datos de entrenamiento, incluyendo ruido y variaciones particulares. Puede producir un error muy bajo en entrenamiento, pero funcionar mal con datos nuevos.

El objetivo no es reproducir perfectamente los datos de entrenamiento, sino encontrar un modelo que **generalice** adecuadamente a observaciones no utilizadas durante el ajuste.
En las figuras siguientes:

- los puntos azules representan datos de entrenamiento;
- los puntos naranjas representan datos de validación;
- la línea gris representa el patrón general que genera los datos;
- la línea roja representa el comportamiento ilustrativo del modelo.

In [ ]:
# Un ejemplo visual.
import numpy as np
import matplotlib.pyplot as plt

# Semilla para obtener siempre la misma figura.
rng = np.random.default_rng(12)

# Patrón general utilizado como referencia.
x_referencia = np.linspace(0, 10, 500)
y_referencia = (
    7
    + 0.45 * x_referencia
    + 1.8 * np.sin(0.9 * x_referencia)
)

# Puntos ilustrativos de entrenamiento.
x_entrenamiento = np.linspace(0.4, 9.6, 14)
y_entrenamiento = (
    7
    + 0.45 * x_entrenamiento
    + 1.8 * np.sin(0.9 * x_entrenamiento)
    + rng.normal(0, 0.55, len(x_entrenamiento))
)

# Puntos ilustrativos de validación.
x_validacion = np.linspace(0.8, 9.2, 7)
y_validacion = (
    7
    + 0.45 * x_validacion
    + 1.8 * np.sin(0.9 * x_validacion)
    + rng.normal(0, 0.55, len(x_validacion))
)

# Línea ilustrativa de underfitting:
# una línea casi recta que no representa la curvatura general.
y_underfitting = 7.4 + 0.35 * x_referencia

# Línea ilustrativa de ajuste adecuado:
# sigue el patrón general sin reproducir el ruido.
y_ajuste_adecuado = (
    7
    + 0.45 * x_referencia
    + 1.8 * np.sin(0.9 * x_referencia)
)

# Línea ilustrativa de overfitting:
# añade oscilaciones rápidas para representar un seguimiento excesivo
# de las variaciones particulares de los datos de entrenamiento.
y_overfitting = (
    7
    + 0.45 * x_referencia
    + 1.8 * np.sin(0.9 * x_referencia)
    + 0.65 * np.sin(6.5 * x_referencia)
    + 0.35 * np.sin(11 * x_referencia)
)

# Guardamos las tres líneas y sus títulos para dibujarlas de forma ordenada.
casos = [
    (
        "Underfitting",
        y_underfitting,
        "Modelo demasiado simple"
    ),
    (
        "Ajuste adecuado",
        y_ajuste_adecuado,
        "Representa el patrón general"
    ),
    (
        "Overfitting",
        y_overfitting,
        "Sigue variaciones demasiado pequeñas"
    )
]

# Creamos tres gráficos con los mismos límites para facilitar la comparación.
figura, ejes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(16, 4.8),
    sharex=True,
    sharey=True
)

for eje, caso in zip(ejes, casos):

    titulo = caso[0]
    linea_modelo = caso[1]
    explicacion = caso[2]

    # Patrón general de referencia.
    eje.plot(
        x_referencia,
        y_referencia,
        color="gray",
        linestyle="--",
        linewidth=2,
        label="Patrón general"
    )

    # Datos de entrenamiento.
    eje.scatter(
        x_entrenamiento,
        y_entrenamiento,
        color="tab:blue",
        s=45,
        label="Entrenamiento",
        zorder=3
    )

    # Datos de validación.
    eje.scatter(
        x_validacion,
        y_validacion,
        color="tab:orange",
        marker="s",
        s=45,
        label="Validación",
        zorder=3
    )

    # Línea conceptual del modelo.
    eje.plot(
        x_referencia,
        linea_modelo,
        color="tab:red",
        linewidth=2.5,
        label="Comportamiento del modelo"
    )

    eje.set_title(f"{titulo}\n{explicacion}")
    eje.set_xlabel("Variable explicativa")

# Etiqueta común del eje vertical.
ejes[0].set_ylabel("Variable que se quiere estimar")

# La leyenda se muestra una sola vez para evitar repetición.
ejes[2].legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

figura.suptitle(
    "Demostración conceptual de underfitting y overfitting",
    fontsize=14
)

plt.tight_layout()
plt.show()

### Actividad M8_8.27

1. Identifica un ausente, un duplicado, un imposible y un valor a revisar.
2. Justifica un tratamiento no destructivo.
4. Compara Pearson y Spearman.
6. Explica una consecuencia de la dependencia espacial o temporal.
7. Propón una división de entrenamiento, validación y prueba.

---
# Referencia rápida

$$
R=x_{max}-x_{min}
$$

$$
IQR=Q_3-Q_1
$$

$$
s^2=\frac{1}{n-1}\sum(x_i-\bar{x})^2
$$

$$
s=\sqrt{s^2}
$$

$$
r=\frac{cov(X,Y)}{s_Xs_Y}
$$

Principios:

- definir la unidad de observación;
- conservar espacio, tiempo, variables, unidades y metadatos;
- crear explícitamente el esquema de la base;
- usar joins mediante claves estables;
- no confundir cero, ausencia y censura;
- no eliminar extremos sin contexto;
- observar gráficos antes de correlaciones;
- comparar resultados globales y por grupos;
- reconocer dependencia espacial y temporal.